In [ ]:
import pandas as pd
from torch.utils.data import (
    Dataset,
)
from transformers import (
    BertTokenizer,
    BertForMaskedLM, 
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments
)

In [2]:
data = pd.read_csv('./corpus.csv')

data.shape

(6953, 2)

In [3]:
sentences = data['combined_data'].tolist()

In [5]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.examples = tokenizer(
            texts, 
            truncation=True, 
            padding='max_length', 
            max_length=max_length, 
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.examples['input_ids'])

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.examples.items()}

In [7]:
tokenizer = BertTokenizer.from_pretrained('rubert_cased_L-12_H-768_A-12_pt_v1')
model = BertForMaskedLM.from_pretrained('rubert_cased_L-12_H-768_A-12_pt_v1')

In [ ]:
# Заморозка всех параметров модели.
for param in model.parameters():
    param.requires_grad = False
    param.data = param.data.contiguous()

# Разморозка параметров слоя эмбеддингов.
for param in model.bert.embeddings.parameters():
    param.requires_grad = True

In [9]:
# Создание датасета.
dataset = TextDataset(sentences, tokenizer)


In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=0.15
    )

In [12]:
# Настройка аргументов обучения
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=10_000,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

In [13]:
trainer.train()

  0%|          | 0/20859 [00:00<?, ?it/s]

{'loss': 2.344, 'grad_norm': 58.81077575683594, 'learning_rate': 4.880147658085239e-05, 'epoch': 0.07}
{'loss': 1.6205, 'grad_norm': 28.674972534179688, 'learning_rate': 4.7602953161704786e-05, 'epoch': 0.14}
{'loss': 1.3681, 'grad_norm': 15.58421802520752, 'learning_rate': 4.640442974255717e-05, 'epoch': 0.22}
{'loss': 1.3062, 'grad_norm': 43.509098052978516, 'learning_rate': 4.520590632340956e-05, 'epoch': 0.29}
{'loss': 1.1401, 'grad_norm': 13.9491548538208, 'learning_rate': 4.400738290426195e-05, 'epoch': 0.36}
{'loss': 1.1829, 'grad_norm': 14.024105072021484, 'learning_rate': 4.280885948511434e-05, 'epoch': 0.43}
{'loss': 1.0625, 'grad_norm': 12.181090354919434, 'learning_rate': 4.161033606596673e-05, 'epoch': 0.5}
{'loss': 1.0483, 'grad_norm': 6.89865255355835, 'learning_rate': 4.041181264681912e-05, 'epoch': 0.58}
{'loss': 0.9947, 'grad_norm': 7.429097652435303, 'learning_rate': 3.9213289227671515e-05, 'epoch': 0.65}
{'loss': 0.8994, 'grad_norm': 15.720064163208008, 'learning_ra

TrainOutput(global_step=20859, training_loss=0.879007401016713, metrics={'train_runtime': 6145.5892, 'train_samples_per_second': 3.394, 'train_steps_per_second': 3.394, 'total_flos': 5495893809638400.0, 'train_loss': 0.879007401016713, 'epoch': 3.0})